# Análisis exploratorio orientado a decisiones

Este notebook estudia aspectos del dataset que cambian una decisión de preparación, evaluación o modelado. No busca acumular gráficos: cada sección responde **qué se encontró** y **qué se hará como consecuencia**.

La lógica reutilizable permanece en `src/eda.py`. El notebook sirve para explicar y ejecutar esa misma lógica.

## 1. Preparación del análisis

Se carga la copia local del dataset Adult y se aplican normalizaciones deterministas: nombres sin espacios, símbolos de faltantes convertidos a nulos y target reducido a las dos categorías esperadas. Todavía no se imputan valores ni se entrena un modelo.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, OUTPUT_DIR, TARGET
from src.data import load_adult
from src.eda import build_eda_decisions

df = load_adult()
print(f"Registros: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
df.head()


Registros: 48,842
Columnas: 15


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 2. Balance de la variable objetivo

Se compara la proporción de personas con ingreso `<=50K` y `>50K`. La clase de ingresos altos es minoritaria, por lo que una exactitud aparentemente buena podría ocultar un desempeño pobre precisamente en el grupo que se quiere identificar.

**Decisión:** conservar la proporción de clases al dividir los datos, equilibrar el peso de las clases durante el entrenamiento y evaluar con F1, recall y ROC-AUC además de accuracy.

In [2]:
target_distribution = df[TARGET].value_counts(normalize=True).mul(100).rename("porcentaje")
target_distribution.to_frame()


,porcentaje
income,
<=50K,76.071823
>50K,23.928177


## 3. Valores faltantes

Se calcula qué parte de cada variable está ausente. El objetivo no es borrar automáticamente las filas, sino medir cuánta información se perdería y elegir una estrategia que pueda reproducirse cuando lleguen datos nuevos.

**Decisión:** conservar los registros e imputar dentro del pipeline. Las estadísticas de imputación se aprenderán únicamente con entrenamiento para evitar que el conjunto de prueba influya en el modelo.

In [3]:
missing_summary = df.drop(columns=TARGET).isna().mean().mul(100)
missing_summary[missing_summary > 0].sort_values(ascending=False).rename("porcentaje_faltante").to_frame()


,porcentaje_faltante
occupation,5.751198
workclass,5.730724
native-country,1.754637


## 4. Educación: consistencia y redundancia

`education` expresa el nivel mediante una categoría y `education-num` mediante un código. Se comprueba que cada categoría tenga un único código. Un máximo igual a uno indica consistencia, aunque también confirma que las variables contienen información estrechamente relacionada.

**Decisión:** conservar ambas en el modelo inicial de árboles y revisar posteriormente su importancia. No se crea una tercera representación equivalente.

In [4]:
education_mapping = (
    df.groupby("education")["education-num"]
      .agg(["min", "max", "nunique", "count"])
      .sort_values("min")
)
education_mapping


,min,max,nunique,count
education,,,,
Preschool,1,1,1,83
1st-4th,2,2,1,247
5th-6th,3,3,1,509
7th-8th,4,4,1,955
9th,5,5,1,756
10th,6,6,1,1389
11th,7,7,1,1812
12th,8,8,1,657
HS-grad,9,9,1,15784


## 5. Ceros en ganancias y pérdidas de capital

La mayoría de las personas tiene cero en `capital-gain` y `capital-loss`. Esos ceros significan que no se reportó una ganancia o pérdida de capital; no representan información ausente. La distribución queda concentrada en cero con pocos valores altos.

**Decisión:** mantener los ceros y usar un modelo capaz de aprender umbrales y distribuciones asimétricas, sin reemplazarlos mediante imputación.

In [5]:
capital_zeros = df[["capital-gain", "capital-loss"]].eq(0).mean().mul(100)
capital_zeros.rename("porcentaje_en_cero").to_frame()


,porcentaje_en_cero
capital-gain,91.738668
capital-loss,95.327792


## 6. Horas trabajadas e ingreso

Las horas semanales se agrupan en bandas para observar cómo cambia la proporción de ingresos mayores a 50K. Las bandas se usan solo para interpretar la relación; el modelo conserva el valor numérico original.

**Decisión:** mantener `hours-per-week` y permitir una relación no lineal. No se supone que aumentar una hora tenga siempre el mismo efecto.

In [6]:
income_by_hours = (
    df.assign(hours_band=pd.cut(df["hours-per-week"], [0, 30, 40, 60, 100]))
      .groupby("hours_band", observed=True)[TARGET]
      .apply(lambda values: values.eq(">50K").mean())
      .mul(100)
)
income_by_hours.rename("porcentaje_mayor_50K").to_frame()


,porcentaje_mayor_50K
hours_band,
"(0, 30]",6.699783
"(30, 40]",20.342355
"(40, 60]",40.572736
"(60, 100]",35.739857


## 7. Generación del entregable del EDA

La función compartida genera los tres gráficos y construye una tabla que conecta cada análisis con su resultado y su decisión. Esta tabla es el entregable principal porque deja documentado cómo el EDA afecta al pipeline.

In [7]:
decisions = build_eda_decisions(df, FIGURES_DIR)
OUTPUT_DIR.mkdir(exist_ok=True)
decisions.to_csv(OUTPUT_DIR / "eda_decisiones.csv", index=False)
decisions


,analisis,resultado,decision
0,Balance de clases,clase minoritaria=23.93%,"Partición estratificada, class_weight='balance..."
1,Valores faltantes,máximo=5.75% en occupation,No eliminar filas; imputar dentro del pipeline...
2,Redundancia education,máximo de códigos por categoría=1,Conservar ambas representaciones para el model...
3,Ceros en variables de capital,gain=91.74%; loss=95.33%,No tratar ceros como faltantes; comparar model...
4,Horas de trabajo,la tasa >50K cambia entre bandas de horas,Conservar hours-per-week y comparar modelos li...


## 8. Conclusión

El análisis conduce a cinco decisiones: proteger la clase minoritaria, imputar sin eliminar filas, vigilar la redundancia educativa, conservar los ceros estructurales de capital y permitir relaciones no lineales para las horas trabajadas. Estas decisiones se implementan posteriormente en el pipeline de características y el Random Forest.